# TechCorp - Fine-tuning LoRA medical sur Colab

Ce notebook utilise les scripts du depot pour lancer un fine-tuning LoRA du dataset medical nettoye, puis generer quelques reponses de test.

Prealable :
- ouvrir le notebook dans Google Colab
- cloner le depot ou monter un Drive qui le contient
- verifier que les fichiers `data-IA/data/dataset/export_lora/train_lora.jsonl` et `validation_lora.jsonl` sont presents

In [ ]:
!nvidia-smi
!pip install -q transformers datasets peft trl accelerate bitsandbytes

In [ ]:
from pathlib import Path
import os

# Adapter cette variable a votre environnement Colab.
# Exemple apres git clone : /content/HACKATHON-IA
REPO_DIR = Path('/content/HACKATHON-IA')

TRAIN_FILE = REPO_DIR / 'data-IA' / 'data' / 'dataset' / 'export_lora' / 'train_lora.jsonl'
EVAL_FILE = REPO_DIR / 'data-IA' / 'data' / 'dataset' / 'export_lora' / 'validation_lora.jsonl'
OUTPUT_DIR = REPO_DIR / 'data-IA' / 'IA' / 'runs' / 'qwen25-medical-lora'
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

print('REPO_DIR  :', REPO_DIR)
print('TRAIN_FILE:', TRAIN_FILE)
print('EVAL_FILE :', EVAL_FILE)
print('OUTPUT_DIR:', OUTPUT_DIR)

assert REPO_DIR.exists(), f'Depot introuvable: {REPO_DIR}'
assert TRAIN_FILE.exists(), f'Fichier train introuvable: {TRAIN_FILE}'
assert EVAL_FILE.exists(), f'Fichier validation introuvable: {EVAL_FILE}'

os.chdir(REPO_DIR)
print('cwd ->', Path.cwd())

In [ ]:
!python data-IA/IA/01_finetune_lora_medical.py \
  --model-name "$MODEL_NAME" \
  --train-file "$TRAIN_FILE" \
  --eval-file "$EVAL_FILE" \
  --output-dir "$OUTPUT_DIR" \
  --num-train-epochs 1 \
  --per-device-train-batch-size 2 \
  --per-device-eval-batch-size 2 \
  --gradient-accumulation-steps 8 \
  --learning-rate 2e-4 \
  --max-seq-length 1024

In [ ]:
!python data-IA/IA/02_generate_medical_samples.py \
  --model-name "$MODEL_NAME" \
  --adapter-path "$OUTPUT_DIR" \
  --max-new-tokens 220

Points d'attention :
- si la VRAM est trop juste, reduire `--max-seq-length` et/ou augmenter `--gradient-accumulation-steps`
- si vous voulez un entrainement plus long, passer `--num-train-epochs` a `2` ou `3`
- ce notebook entraine uniquement le modele medical experimental, pas le modele financier de production